In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"


In [ ]:
import torch
from torch import Generator, Tensor, device, nn
from torch.utils.data import DataLoader, Dataset as TorchDataset
from torch.optim import Adam, SGD, Optimizer
from torch.nn import Parameter

from torchvision.transforms import Compose, InterpolationMode, Resize, ToTensor

import timm
from timm.models.vision_transformer import VisionTransformer

from PIL.Image import Image

from datasets import Dataset as HFDataset, DatasetDict, NamedSplit, load_dataset

import numpy as np

from collections.abc import Iterator
from typing import Any, ClassVar, cast

import random

from enum import Enum

from abc import ABC, abstractmethod

import wandb

import copy


In [ ]:
from kaggle_secrets import UserSecretsClient
wandb.login(key=UserSecretsClient().get_secret("WANDB_API_KEY"))


 # 1. Data

 ## 1.1. Core

In [ ]:
class Batch:
    """
    A batch of data (label included).
    Can carry queries (CLS tokens) after computed via a ViT model.
    """

    IMAGE_SHAPE: ClassVar[tuple[int, int, int]] = (3, 224, 224)

    def __init__(self, X: Tensor, Y: Tensor, queries: Tensor | None = None) -> None:
        self._assert_valid_X(X)
        self._assert_valid_Y(Y)
        self._assert_matching_batch_sizes(X, Y)
        if queries is not None:
            self._assert_valid_queries(queries, X)

        self.X = X
        self.Y = Y
        self.queries = queries

    def to(self, device: device | str) -> "Batch":
        """
        Moves the batch's data to the specified device, used when moving data to a GPU for computation or back to the CPU.


        Args:
            device (device | str): The target device to which data will be moved (example: "cpu", "cuda",...)

        Returns:
            Batch: Current Batch instance with data moved to the specified device.
        """
        self.X = self.X.to(device, non_blocking=True)
        self.Y = self.Y.to(device, non_blocking=True)
        if self.queries is not None:
            self.queries = self.queries.to(device, non_blocking=True)
        return self

    def get_X(self) -> Tensor:
        return self.X

    def get_Y(self) -> Tensor:
        return self.Y

    def get_queries(self) -> Tensor:
        assert self.queries is not None
        return self.queries

    def select(self, indices: Tensor) -> "Batch":
        return Batch(
            self.X[indices],
            self.Y[indices],
            self.get_queries()[indices],
        )

    def pin_memory(self) -> "Batch":
        self.X = self.X.pin_memory()
        self.Y = self.Y.pin_memory()
        if self.queries is not None:
            self.queries = self.queries.pin_memory()
        return self

    def get_batch_size(self) -> int:
        return self.Y.shape[0]

    def merge(self, other: "Batch") -> "Batch":
        """
        Merges the current instance with another Batch instance after moving it to the same device as the current.

        Args:
            other (Batch): Other Batch instance.

        Returns:
            Batch: Merged Batch instance.
        """

        other_X = other.get_X().to(self.X.device, non_blocking=True)
        other_Y = other.get_Y().to(self.Y.device, non_blocking=True)
        other_queries = other.get_queries().to(
            self.get_queries().device, non_blocking=True
        )

        merged_X = torch.cat((self.X, other_X), dim=0)
        merged_Y = torch.cat((self.Y, other_Y), dim=0)
        merged_queries = torch.cat((self.get_queries(), other_queries), dim=0)

        return Batch(
            X=merged_X,
            Y=merged_Y,
            queries=merged_queries,
        )

    def _assert_valid_X(self, X: Tensor) -> None:
        assert X.ndim == 4
        assert tuple(X.shape[1:]) == self.IMAGE_SHAPE
        assert X.dtype == torch.float32

    def _assert_valid_Y(self, Y: Tensor) -> None:
        assert Y.ndim == 1
        assert Y.dtype == torch.long

    def _assert_matching_batch_sizes(self, X: Tensor, Y: Tensor) -> None:
        assert X.shape[0] == Y.shape[0]

    def _assert_valid_queries(self, queries: Tensor, X: Tensor) -> None:
        assert queries.ndim == 2
        assert queries.shape[0] == X.shape[0]
        assert queries.dtype == torch.float32



In [ ]:
class TorchDatasetAdapter(TorchDataset[dict[str, Any]]):
    """
    An adapter that makes a Hugging Face Dataset object compatible with a Pytorch Dataset object.
    """

    def __init__(self, hf_dataset: HFDataset, queries: Tensor | None = None) -> None:
        self.hf_dataset = hf_dataset
        self.queries = queries

    def __len__(self) -> int:
        return len(self.hf_dataset)

    def __getitem__(self, index: int) -> dict[str, Any]:
        item = self.hf_dataset[index]

        if self.queries is not None:
            item["queries"] = self.queries[index]

        return item


class BatchIterator:
    """
    Packages batches inside of one single task into an iterable for easy processing. Wraps Pytorch's DataLoader object.
    """

    def __init__(
        self,
        dataset: TorchDatasetAdapter,
        batch_size: int,
        shuffle: bool,
        random_seed: int,
        num_workers: int = 4,
    ) -> None:
        """
        Creates a DataLoader with the necessary information. Pinned CPU memory is used to allow for more efficient CPU-GPU memory transfer.

        Args:
            dataset (TorchDatasetAdapter): Dataset.
            batch_size (int): Batch size.
            shuffle (bool): Toggle shuffling, the DataLoader picks data points at random to make batches.
            random_seed (int): Random seed.
            num_workers (int, optional): Number of subprocesses used for loading data.
        """
        assert num_workers > 0

        generator = Generator().manual_seed(random_seed)

        # Shuffling orders start at the same point with the same random seed
        self.data_loader = DataLoader(
            dataset=dataset,
            batch_size=batch_size,
            shuffle=shuffle,
            num_workers=num_workers,
            pin_memory=True,
            collate_fn=self._collate_fn,
            generator=generator,
        )

    @staticmethod
    def _collate_fn(raw_batch: list[dict[str, Any]]) -> Batch:
        """
        Packages a specified number (batch size) of data points into a batch, transforming them into a Batch object. Used to initialize the DataLoader.

        Args:
            raw_batch (list[dict[str, Any]]): Raw batch data points.

        Returns:
            Batch: A packaged Batch object.
        """
        x_array = [item["X"] for item in raw_batch]
        y_array = [item["Y"] for item in raw_batch]

        queries = None
        if "queries" in raw_batch[0]:
            queries = torch.stack([item["queries"] for item in raw_batch])

        return Batch(
            X=torch.stack(x_array),
            Y=torch.tensor(y_array, dtype=torch.long),
            queries=queries,
        )

    def __iter__(self) -> Iterator[Batch]:
        return iter(self.data_loader)

    def __len__(self) -> int:
        return len(self.data_loader)



 ## 1.2. Tasks

In [ ]:
class ConvertChannels:
    """
    Utility class for converting the channels of an image to the desired mode.
    """

    def __init__(self, mode: str) -> None:
        self.mode = mode

    def __call__(self, image: Image) -> Image:
        if image.mode != self.mode:
            return image.convert(self.mode)
        return image

class Task:
    """
    A task of train and test data. Returns data in the form of batches via a BatchIterator object.
    Data inside a task always have a "train" and a "test" split. Data inside a split always has "X" and "Y" columns. This is ensured through preprocessing after fetching and validation at task initialization.
    In task-agnostic settings. The entire dataset is crammed into one single task.
    """

    REQUIRED_SPLITS: ClassVar[tuple[str, ...]] = ("train", "test")
    REQUIRED_COLUMNS: ClassVar[tuple[str, ...]] = ("X", "Y")

    IMAGE_SHAPE: ClassVar[tuple[int, int]] = (224, 224)
    # NOTE: Paper did not specify which interpolation mode to use
    RESIZE_INTERPOLATION_MODE: ClassVar[InterpolationMode] = InterpolationMode.BILINEAR

    def __init__(
        self,
        class_ids: list[int],
        data: DatasetDict,
        random_seed: int,
    ) -> None:
        self._assert_required_splits_exist(data)
        self._assert_required_columns_exist(data)

        self.class_ids = class_ids
        self.random_seed = random_seed
        self.transform = self._create_data_point_transform_pipeline()
        self.data = data

        self.queries_by_split: dict[str, Tensor] = {}

    def set_queries(self, split: str, queries: Tensor) -> None:
        assert queries.shape[0] == len(self.data[split])
        self.queries_by_split[split] = queries

    def get_class_ids(self) -> list[int]:
        return self.class_ids

    def get_train_batches(self, batch_size: int) -> BatchIterator:
        return self.get_batches("train", batch_size, shuffle=True)

    def get_test_batches(self, batch_size: int) -> BatchIterator:
        return self.get_batches("test", batch_size, shuffle=False)

    def get_batches(self, split: str, batch_size: int, shuffle: bool) -> BatchIterator:
        dataset = self.data[split].with_transform(self._apply_transform_to_batch)
        adapter = TorchDatasetAdapter(dataset, self.queries_by_split.get(split))

        return BatchIterator(adapter, batch_size, shuffle, self.random_seed)

    def get_n_samples_per_class(self, samples_per_class: int) -> Batch:
        """
        Yields a fixed number of samples per class from the data and packages them into a batch before returning.

        Args:
            samples_per_class (int): Number of samples per class.

        Returns:
            Batch: Packaged preprocessed samples.
        """
        train_dataset = self.data["train"]

        labels = np.asarray(train_dataset["Y"])
        selected_indices = self._sample_indices_of_every_class(
            labels, samples_per_class
        )
        raw_samples = train_dataset.select(selected_indices)[:]
        transformed_samples = self._apply_transform_to_batch(raw_samples)
        queries = self.queries_by_split["train"][selected_indices]

        return Batch(
            X=torch.stack(transformed_samples["X"]),
            Y=torch.tensor(transformed_samples["Y"], dtype=torch.long),
            queries=queries,
        )

    def _sample_indices_of_every_class(
        self, labels_column: np.ndarray, samples_per_class: int
    ) -> list[int]:
        rng = np.random.default_rng(self.random_seed)

        selected_indices: list[int] = []
        for class_id in self.class_ids:
            selected_indices.extend(
                self._sample_indices_of_class(
                    labels_column,
                    class_id,
                    samples_per_class,
                    rng,
                )
            )

        return selected_indices

    def _sample_indices_of_class(
        self,
        labels_column: np.ndarray,
        class_id: int,
        samples_per_class: int,
        rng: np.random.Generator,
    ) -> list[int]:
        indices_of_matching_instances = np.flatnonzero(labels_column == class_id)
        assert len(indices_of_matching_instances) >= samples_per_class

        return rng.choice(
            indices_of_matching_instances, size=samples_per_class, replace=False
        ).tolist()

    def _apply_transform_to_batch(
        self, raw_batch: dict[str, list[Any]]
    ) -> dict[str, list[Any]]:
        """
        Transforms a batch of data, preprocessing each data point's features and label.

        Args:
            raw_batch (dict[str, list[Any]]): Raw batch of data.

        Returns:
            dict[str, list[Any]]: Transformed batch of data.
        """
        raw_X = raw_batch["X"]
        raw_Y = raw_batch["Y"]

        processed_X = [self._preprocess_data_point(data_point) for data_point in raw_X]
        processed_Y = [self._preprocess_label(label) for label in raw_Y]

        return {
            "X": processed_X,
            "Y": processed_Y,
        }

    def _preprocess_data_point(self, data_point: Image) -> Tensor:
        """
        Applies the transform pipeline to a datapoint's features (see detailed info at _create_data_point_transform_pipeline).

        Args:
            data_point (Image): Data point features.

        Returns:
            Tensor: Transformed data point features.
        """
        transformed_data_point = self.transform(data_point)
        assert isinstance(transformed_data_point, Tensor)

        return transformed_data_point

    def _preprocess_label(self, label: int) -> int:
        """
        Transforms a data point label. Does nothing for most datasets except for 5-datasets, where it applies an offset on the labels.

        Args:
            label (int): Label.

        Returns:
            int: Transformed label.
        """
        return label

    def _create_data_point_transform_pipeline(self) -> Compose:
        """
        Lazy transformation pipeline, including: converting channels to RGB, resizing image to the correct size (224 x 224), converting image to a Pytorch tensor while also changing layout from HWC to CHW.

        Returns:
            Compose: Returned pipeline.
        """
        pipeline = Compose(
            [
                ConvertChannels("RGB"),
                Resize(self.IMAGE_SHAPE, interpolation=self.RESIZE_INTERPOLATION_MODE),
                ToTensor(),
            ]
        )

        return pipeline

    def _assert_required_splits_exist(self, data: DatasetDict) -> None:
        for split_name in self.REQUIRED_SPLITS:
            assert split_name in data

    def _assert_required_columns_exist(self, data: DatasetDict) -> None:
        for split_name in self.REQUIRED_SPLITS:
            split_columns = data[split_name].column_names

            for column_name in self.REQUIRED_COLUMNS:
                assert column_name in split_columns



In [ ]:
class CIFARTask(Task):
    """
    A task class designed for the CIFAR-100 dataset.
    Information already covered in the docstrings inside the base Task class won't be specified again.
    """

    pass



In [ ]:
class CORe50Task(Task):
    """
    A task class designed for the CORe50 dataset.
    Information already covered in the docstrings inside the base Task class won't be specified again.
    """

    pass



In [ ]:
class FiveDatasetsTask(Task):
    """
    A task class designed for the 5-datasets.
    Information already covered in the docstrings inside the base Task class won't be specified again.
    """

    def __init__(
        self,
        class_ids: list[int],
        data: DatasetDict,
        random_seed: int,
        label_offset: int,
    ) -> None:
        super().__init__(class_ids, data, random_seed)
        self.label_offset = label_offset

    def _preprocess_label(self, label: int) -> int:
        return label + self.label_offset

    def _sample_indices_of_class(
        self,
        labels_column: np.ndarray,
        class_id: int,
        samples_per_class: int,
        rng: np.random.Generator,
    ) -> list[int]:
        return super()._sample_indices_of_class(
            labels_column,
            class_id - self.label_offset,
            samples_per_class,
            rng,
        )



In [ ]:
class GaussianCIFARTask(CIFARTask):
    """
    A task class designed for the CIFAR-100 dataset (Gaussian version).
    Information already covered in the docstrings inside the base Task class won't be specified again.
    """

    def get_train_batches(self, batch_size: int) -> BatchIterator:
        """
        Similar to the base class get_train_batches, except shuffling is always turned off to preserve Gaussian scheduled data stream order.

        Args:
            batch_size (int): Batch size.

        Returns:
            BatchIterator: Returned batches.
        """
        return self.get_batches("train", batch_size, shuffle=False)


 ## 1.3. Data package

In [ ]:
class DataPackage:
    """
    Contains all the training data of a dataset and its number of classes.
    Training data is divided into tasks. Each task contains its own train and test data. Tasks then yield their data through a BatchIterator object with customizable batch size.
    """

    def __init__(self, tasks: list[Task], num_classes: int) -> None:
        self.tasks = tasks
        self.num_classes = num_classes

    def get_task(self, task_index: int) -> Task:
        return self.tasks[task_index]

    def get_tasks(self) -> list[Task]:
        return self.tasks

    def get_number_of_tasks(self) -> int:
        return len(self.tasks)

    def get_number_of_classes(self) -> int:
        return self.num_classes



 ## 1.4. Fetchers

In [ ]:
class FetcherType(str, Enum):
    CIFAR100 = "CIFAR100"
    GAUSSIAN_CIFAR100 = "GAUSSIAN_CIFAR100"
    CORE50 = "CORE50"
    FIVE_DATASETS = "FIVE_DATASETS"


In [ ]:
class Fetcher(ABC):
    """
    Classes that inherit this class are meant to be used as interfaces to fetch different datasets from different external sources.
    The data is preprocessed to ensure it follows certain contracts established inside this program (see docstring in _preprocess), before being loaded into a DataPackage object to be returned.
    """

    def __init__(self, random_seed: int) -> None:
        self.random_seed = random_seed

    def get_data(self) -> DataPackage:
        """
        Fetches and preprocesses data into a DataPackage before returning.

        Returns:
            DataPackage: Object containing all the necessary information inside the fetched dataset.
        """
        data = self._fetch_data()
        preprocessed_data = self._preprocess(data)

        return preprocessed_data

    @abstractmethod
    def _fetch_data(self) -> Any:
        """
        Fetches dataset from different sources, depending on the child class.

        Returns:
            Any: The raw data returned right after fetching from the external source.
        """
        pass

    @abstractmethod
    def _preprocess(self, data: Any) -> DataPackage:
        """
        Preprocesses data into a DataPackage. The resulting data will always have the two splits: "train" and "test". Each split always has the columns "X", "Y" corresponding to features and labels.
        Different datasets will also have different preprocessing steps.

        Args:
            data (Any): The raw data from the _fetch_data function.

        Returns:
            DataPackage: The preprocessed data packaged into a DataPackage object.
        """
        pass



In [ ]:
class CIFARFetcher(Fetcher):
    """
    Fetcher fetching data for the CIFAR-100 dataset.
    Information already covered in the docstrings inside the base Fetcher class won't be specified again.
    """

    NUM_CLASSES: ClassVar[int] = 100
    NUM_TASKS: ClassVar[int] = 10

    def _fetch_data(self) -> DatasetDict:
        dataset = load_dataset("uoft-cs/cifar100")
        return dataset

    def _preprocess(self, data: DatasetDict) -> DataPackage:
        """
        Preprocesses data by shuffling the 100 class labels, before dividing them into 10 sets, corresponding to 10 tasks. Train and test data inside each task is chosen according to the shuffled class labels inside that task.

        Args:
            data (DatasetDict): Raw fetched data.

        Returns:
            DataPackage: Preprocessed data.
        """
        shuffled_class_ids = self._generate_shuffled_class_ids()
        task_segmented_class_ids = self._perform_class_ids_segmentation_by_task(
            shuffled_class_ids
        )

        tasks = []

        normalized_data = self._normalize_columns(data)

        for class_ids in task_segmented_class_ids:
            filtered_task_data = self._filter_data_by_class_ids(
                normalized_data, class_ids
            )

            task = self._build_task_from_data(filtered_task_data, class_ids)

            tasks.append(task)

        package = self._build_package_from_tasks(tasks)

        return package

    def _generate_shuffled_class_ids(self) -> list[int]:
        """
        Shuffles the 100 class labels.

        Returns:
            list[int]: The shuffled class labels.
        """
        ordered_class_ids = list(range(self.NUM_CLASSES))
        random_generator = random.Random(self.random_seed)
        shuffled_class_ids = random_generator.sample(
            ordered_class_ids, self.NUM_CLASSES
        )

        return shuffled_class_ids

    def _perform_class_ids_segmentation_by_task(
        self, shuffled_class_ids: list[int]
    ) -> list[list[int]]:
        """
        Divides the shuffled class labels into 10 different sets.

        Args:
            shuffled_class_ids (list[int]): Shuffled class labels.

        Returns:
            list[list[int]]: An array of 10 distinct sets of labels.
        """
        classes_per_task = self.NUM_CLASSES // self.NUM_TASKS
        segmented_class_ids = []

        for task_index in range(self.NUM_TASKS):
            start_index = task_index * classes_per_task
            end_index = start_index + classes_per_task

            current_task_class_ids = shuffled_class_ids[start_index:end_index]

            segmented_class_ids.append(current_task_class_ids)

        return segmented_class_ids

    def _normalize_columns(self, data: DatasetDict) -> DatasetDict:
        return data.rename_columns({"img": "X", "fine_label": "Y"}).select_columns(
            ["X", "Y"]
        )

    def _filter_data_by_class_ids(
        self, data: DatasetDict, class_ids: list[int]
    ) -> DatasetDict:
        """
        Filters data points for one task according to a set of shuffled class labels.

        Args:
            data (DatasetDict): Dataset.
            class_ids (list[int]): A single set of shuffled class labels.

        Returns:
            DatasetDict: Chosen data points.
        """
        filtered_data: dict[str | NamedSplit, HFDataset] = {}

        for split_name, dataset in data.items():
            labels = np.asarray(dataset["Y"])
            selected_instances_bool = np.isin(labels, class_ids)
            selected_instances_indices = np.flatnonzero(selected_instances_bool)

            filtered_data[split_name] = dataset.select(selected_instances_indices)

        return DatasetDict(filtered_data)

    def _build_task_from_data(self, data: DatasetDict, class_ids: list[int]) -> Task:
        return CIFARTask(
            class_ids=class_ids,
            data=data,
            random_seed=self.random_seed,
        )

    def _build_package_from_tasks(self, tasks: list[Task]) -> DataPackage:
        return DataPackage(
            tasks=tasks,
            num_classes=self.NUM_CLASSES,
        )



In [ ]:
class CORe50Fetcher(Fetcher):
    """
    Fetcher fetching data for the CORe50 dataset.
    Information already covered in the docstrings inside the base Fetcher class won't be specified again.
    """

    NUM_CLASSES: ClassVar[int] = 50
    TRAIN_SESSIONS: ClassVar[list[int]] = [1, 2, 4, 5, 6, 8, 9, 11]
    TEST_SESSIONS: ClassVar[list[int]] = [3, 7, 10]

    def _fetch_data(self) -> DatasetDict:
        return load_dataset("nhuy165/core50-sessions")

    def _preprocess(self, data: DatasetDict) -> DataPackage:
        """
        Preprocesses data by choosing data points based on the session indices predefined by the dataset. Indices range from 1 to 11 (inclusive), with indices 3, 7, 10 being used for the test data and the rest are used for training.
        Each task contains different training data corresponding to one session index. All tasks contain the same test data (session 3, 7, 10).

        Args:
            data (DatasetDict): Raw fetched data.

        Returns:
            DataPackage: Preprocessed data.
        """
        normalized_data = self._normalize_columns(data)
        full_dataset = normalized_data["train"]  # HF dataset only has 1 split

        test_dataset = self._extract_test_dataset(full_dataset)

        tasks = []

        for session_id in self.TRAIN_SESSIONS:
            train_dataset = self._filter_data_by_session_id(full_dataset, session_id)

            dataset_dict = DatasetDict(
                {
                    "train": train_dataset,
                    "test": test_dataset,
                }
            )

            task = self._build_task_from_data(dataset_dict)
            tasks.append(task)

        return self._build_package_from_tasks(tasks)

    def _normalize_columns(self, data: DatasetDict) -> DatasetDict:
        return data.rename_columns({"image": "X", "label": "Y"})

    def _extract_test_dataset(self, dataset: HFDataset) -> HFDataset:
        """
        Filters test data points for all tasks, chosen data points have session indices of 3, 7 or 10.

        Args:
            dataset (HFDataset): Dataset.

        Returns:
            HFDataset: Chosen data points.
        """
        sessions = np.asarray(dataset["session"])
        selected_instances_bool = np.isin(sessions, self.TEST_SESSIONS)
        selected_instances_indices = np.flatnonzero(selected_instances_bool)

        return dataset.select(selected_instances_indices)

    def _filter_data_by_session_id(self, dataset: HFDataset, session_id: int) -> HFDataset:
        """
        Filters training data points for one task according to the session id.

        Args:
            data (HFDataset): Dataset.
            session_id (int): Session id.

        Returns:
            DatasetDict: Chosen data points.
        """
        sessions = np.asarray(dataset["session"])
        selected_instances_bool = sessions == session_id
        selected_instances_indices = np.flatnonzero(selected_instances_bool)

        return dataset.select(selected_instances_indices)

    def _build_task_from_data(self, data: DatasetDict) -> Task:
        class_ids = list(range(self.NUM_CLASSES))

        return CORe50Task(
            class_ids=class_ids,
            data=data,
            random_seed=self.random_seed,
        )

    def _build_package_from_tasks(self, tasks: list[Task]) -> DataPackage:
        return DataPackage(
            tasks=tasks,
            num_classes=self.NUM_CLASSES,
        )



In [ ]:
class FiveDatasetsFetcher(Fetcher):
    """
    Fetcher fetching data for the 5-datasets.
    Information already covered in the docstrings inside the base Fetcher class won't be specified again.
    """

    NUM_CLASSES_PER_TASK: ClassVar[int] = 10
    NUM_TASKS: ClassVar[int] = 5

    def _fetch_data(self) -> dict[str, DatasetDict]:
        return {
            "cifar10": load_dataset("uoft-cs/cifar10"),
            "mnist": load_dataset("ylecun/mnist"),
            "fashion_mnist": load_dataset("zalando-datasets/fashion_mnist"),
            "svhn": load_dataset("ufldl-stanford/svhn", "cropped_digits"),
            "notmnist": load_dataset("anubhavmaity/notMNIST"),
        }

    def _preprocess(self, data: dict[str, DatasetDict]) -> DataPackage:
        """
        Preprocesses data by generating distinct class labels for the five datasets. Each dataset corresponds to one task. The current datasets ordering is: CIFAR10, MNIST, Fashion-MNIST, SVHN, notMNIST (this order is arbitrary and was not explicitly stated in the original paper).

        Args:
            data (dict[str, DatasetDict]): Raw fetched data.

        Returns:
            DataPackage: Preprocessed data.
        """
        normalized_data = self._normalize_columns_and_splits(data)

        tasks = []
        global_offset = 0

        for dataset in normalized_data.values():
            class_ids = self._generate_class_ids(global_offset)

            task = self._build_task_from_data(dataset, class_ids, global_offset)
            tasks.append(task)

            global_offset += self.NUM_CLASSES_PER_TASK

        return self._build_package_from_tasks(tasks)

    def _normalize_columns_and_splits(
        self, data: dict[str, DatasetDict]
    ) -> dict[str, DatasetDict]:
        old_column_names_map: dict[str, tuple[str, str]] = {
            "cifar10": ("img", "label"),
            "mnist": ("image", "label"),
            "fashion_mnist": ("image", "label"),
            "svhn": ("image", "label"),
            "notmnist": ("image", "label"),
        }

        for dataset_name in data:
            X_old_name, Y_old_name = old_column_names_map[dataset_name]

            data[dataset_name] = (
                data[dataset_name]
                .rename_columns(
                    {
                        X_old_name: "X",
                        Y_old_name: "Y",
                    }
                )
                .select_columns(["X", "Y"])
            )
            if "extra" in data[dataset_name]:
                del data[dataset_name]["extra"]

        return data

    def _generate_class_ids(self, offset: int) -> list[int]:
        """
        Generates distinct class ids while keeping track of a global offset so that the labels from the different datasets don't overlap.

        Args:
            offset (int): Global offset.

        Returns:
            list[int]: Generated class labels.
        """
        return list(range(offset, offset + self.NUM_CLASSES_PER_TASK))

    def _build_task_from_data(
        self, data: DatasetDict, class_ids: list[int], offset: int
    ) -> Task:
        return FiveDatasetsTask(
            class_ids=class_ids,
            data=data,
            random_seed=self.random_seed,
            label_offset=offset,
        )

    def _build_package_from_tasks(self, tasks: list[Task]) -> DataPackage:
        return DataPackage(
            tasks=tasks,
            num_classes=self.NUM_CLASSES_PER_TASK * self.NUM_TASKS,
        )



In [ ]:
class GaussianCIFARFetcher(CIFARFetcher):
    """
    Fetcher fetching data for the CIFAR-100 dataset and applying Gaussian schedule on the data.
    Information already covered in the docstrings inside the base Fetcher class and the CIFARFetcher class won't be specified again.
    """

    APPEARANCE_TIME_STD: ClassVar[float] = 0.05

    def _preprocess(self, data: DatasetDict) -> DataPackage:
        """
        Preprocesses data by applying Gaussian schedule on the data (see more information in _apply_gaussian_schedule). This dataset does not have task boundaries (the entire dataset is one giant task) and is meant to be used in task-agnostic training.

        Args:
            data (DatasetDict): Raw fetched data.

        Returns:
            DataPackage: Preprocessed data.
        """

        normalized_data = self._normalize_columns(data)

        scheduled_train_dataset = self._apply_gaussian_schedule(
            normalized_data["train"]
        )

        normalized_scheduled_data = DatasetDict(
            {
                "train": scheduled_train_dataset,
                "test": normalized_data["test"],
            }
        )

        task = self._build_task_from_data(normalized_scheduled_data)

        package = self._build_package_from_tasks([task])

        return package

    def _apply_gaussian_schedule(self, dataset: HFDataset) -> HFDataset:
        """
        Applies Gaussian schedule to create a continuous data stream where class labels shift slowly.
        Each class label is allocated a peak appearance time (distributed evenly between 0 and 1). Each data point is sampled a specific timestamp using a Gaussian distribution centered around this peak time with a standard deviation specified in APPEARANCE_TIME_STD.

        Args:
            dataset (HFDataset): Dataset.

        Returns:
            HFDataset: Dataset after applied Gaussian schedule.
        """
        random_generator = random.Random(self.random_seed)

        appearance_time_array = []

        for label in dataset["Y"]:
            # Class appearances peak evenly between 0.0 and 1.0
            label_peak_appearance_time = label / max(1, self.NUM_CLASSES - 1)

            sampled_appearance_time = random_generator.gauss(
                label_peak_appearance_time, self.APPEARANCE_TIME_STD
            )

            appearance_time_array.append(sampled_appearance_time)

        dataset_with_time = dataset.add_column("appearance_time", appearance_time_array)
        dataset_scheduled_by_classes = dataset_with_time.sort(
            "appearance_time"
        ).remove_columns(["appearance_time"])

        return dataset_scheduled_by_classes

    def _build_task_from_data(self, data: DatasetDict) -> Task:
        class_ids = list(range(self.NUM_CLASSES))

        return GaussianCIFARTask(
            class_ids=class_ids,
            data=data,
            random_seed=self.random_seed,
        )



 # 2. Model

 ## 2.1. Forward

In [ ]:
class ClassificationHead(nn.Module):
    """
    Performs classification using a transformed output prompt tokens.
    """

    def __init__(
        self,
        embed_dimension: int,
        num_classes: int,
        prompt_length: int,
        top_n: int,
    ) -> None:
        super().__init__()

        self.num_prompt_tokens = prompt_length * top_n
        self.classifier = nn.Linear(embed_dimension, num_classes)

    def extract_transformed_prompt_tokens_and_classify(
        self, transformer_output: Tensor
    ) -> Tensor:
        """
        Extracts prompt tokens from a transformed output and use them to perform classification.

        Args:
            transformer_output (Tensor): Transformed output containing transformed embedded tokens, CLS tokens and prompt tokens.

        Returns:
            Tensor: Logits containing classification results.
        """
        prompt_tokens = self._extract_prompt_tokens(transformer_output)
        averaged_features = self._average_tokens(prompt_tokens)
        logits = self._classify_features(averaged_features)

        return logits

    def _extract_prompt_tokens(self, transformer_output: Tensor) -> Tensor:
        return transformer_output[:, : self.num_prompt_tokens, :]

    def _average_tokens(self, prompt_tokens: Tensor) -> Tensor:
        return prompt_tokens.mean(dim=1)

    def _classify_features(self, features: Tensor) -> Tensor:
        return self.classifier(features)



In [ ]:
class PromptPool(nn.Module):
    """
    Manages a prompt pool and frequency table.
    """

    prompt_frequencies_batch: Tensor
    prompt_normalized_frequencies_task: Tensor

    def __init__(
        self,
        pool_size: int,
        prompt_length: int,
        embed_dimension: int,
        key_dimension: int,
        top_n: int,
        use_frequency_penalty: bool,
    ) -> None:
        super().__init__()

        self.pool_size = pool_size
        self.prompt_length = prompt_length
        self.embed_dimension = embed_dimension
        self.key_dimension = key_dimension
        self.top_n = top_n
        self.use_frequency_penalty = use_frequency_penalty

        self.prompts = nn.Parameter(
            torch.empty(pool_size, prompt_length, embed_dimension)
        )
        self.keys = nn.Parameter(torch.empty(pool_size, key_dimension))

        nn.init.uniform_(self.prompts, -1.0, 1.0)
        nn.init.uniform_(self.keys, -1.0, 1.0)

        self.has_frequency_history = False

        if self.use_frequency_penalty:
            self.register_buffer("prompt_frequencies_batch", torch.zeros(pool_size))
            self.register_buffer(
                "prompt_normalized_frequencies_task", torch.zeros(pool_size)
            )

    def select_prompts_and_calculate_loss_from_queries(
        self, queries: Tensor
    ) -> tuple[Tensor, Tensor]:
        """
        Fetches prompts based on queries (CLS tokens) and calculates surrogate loss in the process.

        Args:
            queries (Tensor): Queries.

        Returns:
            tuple[Tensor, Tensor]: Prompt tokens and surrogate loss.
        """
        with torch.autocast("cuda", enabled=False):
            queries = queries.float()
            with torch.no_grad():
                similarity_matrix = self._calculate_cosine_similarity(
                    queries, self.keys
                )

                selected_prompt_indices = self._get_top_n_prompt_indices(
                    similarity_matrix
                )

                if self.training and self.use_frequency_penalty:
                    self._update_frequency_after_batch(selected_prompt_indices)

            prompt_tokens = self._extract_prompts(selected_prompt_indices)
            surrogate_loss = self._calculate_surrogate_loss(
                queries, selected_prompt_indices
            )

        return prompt_tokens, surrogate_loss

    def update_frequency_after_task(self) -> None:
        """
        Updates prompt frequency table at the end of a task. The new updated frequency is then used for future prompt selection.
        """
        if not self.use_frequency_penalty:
            raise Exception(
                "Prompt frequency update called when training configured to not use frequency penalty."
            )

        total_prompt_counts: Tensor = self.prompt_frequencies_batch.sum()

        if total_prompt_counts > 0:
            normalized_frequencies = self.prompt_frequencies_batch / total_prompt_counts
            self.prompt_normalized_frequencies_task = normalized_frequencies
            self.has_frequency_history = True

    def _calculate_cosine_similarity(self, queries: Tensor, keys: Tensor) -> Tensor:
        """
        Calculates a cosine similarity matrix of each query to each prompt key.

        Args:
            queries (Tensor): Queries.
            keys (Tensor): Prompt keys.

        Returns:
            Tensor: Similarity matrix.
        """
        normalized_queries = nn.functional.normalize(queries, p=2, dim=1)
        normalized_keys = nn.functional.normalize(keys, p=2, dim=1)
        similarity_matrix = torch.matmul(normalized_queries, normalized_keys.T)

        return similarity_matrix

    def _get_top_n_prompt_indices(self, similarity_matrix: Tensor) -> Tensor:
        """
        Fetches the top most appropriate prompts based on the input similarity matrix. Frequency penalty may be used at this task during training.

        Args:
            similarity_matrix (Tensor): Similarity matrix.

        Returns:
            Tensor: Select prompts' indices.
        """
        cosine_distances = 1.0 - similarity_matrix

        use_penalty = (
            self.training and self.use_frequency_penalty and self.has_frequency_history
        )

        if use_penalty:
            final_distances = cosine_distances * (
                1.0 + self.prompt_normalized_frequencies_task
            )
        else:
            final_distances = cosine_distances

        _, selected_indices = torch.topk(
            final_distances, k=self.top_n, dim=1, largest=False
        )

        return selected_indices

    def _extract_prompts(self, selected_indices: Tensor) -> Tensor:
        selected_prompts = self.prompts[selected_indices]
        batch_size = selected_indices.shape[0]

        return selected_prompts.view(batch_size, -1, self.embed_dimension)

    def _calculate_surrogate_loss(
        self, queries: Tensor, selected_indices: Tensor
    ) -> Tensor:
        """
        Calculates surrogate loss based on queries and selected prompts.

        Args:
            queries (Tensor): Queries.
            selected_indices (Tensor): Selected prompt indices.

        Returns:
            Tensor: Surrogate loss.
        """
        selected_keys = self.keys[selected_indices]

        queries_expanded = queries.unsqueeze(1)

        normalized_queries = nn.functional.normalize(queries_expanded, p=2, dim=2)
        normalized_keys = nn.functional.normalize(selected_keys, p=2, dim=2)

        cosine_similarities = (normalized_queries * normalized_keys).sum(dim=2)
        cosine_distances = 1.0 - cosine_similarities

        return cosine_distances.sum(dim=1).mean()

    def _update_frequency_after_batch(self, selected_indices: Tensor) -> None:
        """
        Updates prompt frequency table at the end of a batch. The updates at this step only serve to record the frequency, the changes do not take effect until the end of the current task.
        """
        flattened_indices = selected_indices.flatten()
        index_counts = torch.bincount(flattened_indices, minlength=self.pool_size)

        self.prompt_frequencies_batch += index_counts


In [ ]:
class ViTFeatureExtractor(nn.Module):
    """
    Holds a pretrained Vision Transformer (ViT).
    Main functionalities include different stages of a model forward class, including: embedding tokens, transforming tokens and extracting CLS tokens after both of these processes.
    """

    def __init__(self, model_name: str = "vit_base_patch16_224.orig_in21k") -> None:
        super().__init__()

        self.vit = cast(
            VisionTransformer,
            timm.create_model(model_name, pretrained=True, num_classes=0),
        )

        self.vit.eval()
        for param in self.vit.parameters():
            param.requires_grad = False

    def pass_through_model_and_extract_cls_tokens(self, X: Tensor) -> Tensor:
        """
        Passes the image through the whole model before extracting the CLS tokens and returning.

        Args:
            X (Tensor): Input features.

        Returns:
            Tensor: CLS tokens.
        """
        with torch.no_grad():
            input_tokens = self.embed_and_build_input_tokens(X)
            output_tokens = self.transform_and_normalize_tokens(input_tokens)

        return output_tokens[:, 0]

    def embed_and_build_input_tokens(self, X: Tensor) -> Tensor:
        """
        Passes the image through the embedding layers, then prepends CLS tokens and adds positional embeddings to the result tokens before returning.

        Args:
            X (Tensor): Input features.

        Returns:
            Tensor: Embedded tokens with prepended CLS and positional embeddings.
        """
        patch_tokens = self._get_patch_tokens(X)
        patch_tokens_with_cls = self._prepend_cls_token(patch_tokens)
        input_tokens = self._add_positional_embeddings(patch_tokens_with_cls)

        return input_tokens

    def transform_and_normalize_tokens(self, input_tokens: Tensor) -> Tensor:
        """
        Passes the embedded tokens with CLS through the transformer layer of the model, before normalizing and returning.

        Args:
            input_tokens (Tensor): Embedded tokens with CLS.

        Returns:
            Tensor: Normalized transformed tokens.
        """
        transformed_tokens = self._pass_through_transformer(input_tokens)
        output_tokens = self._normalize_tokens(transformed_tokens)

        return output_tokens

    def train(self, mode: bool = True) -> "ViTFeatureExtractor":
        """
        Configures the model training mode. Turns on the appropriate mode for all other modules while ensuring the core model is always in evaluation mode.

        Args:
            mode (bool, optional): Turns on training if True.

        Returns:
            ViTFeatureExtractor: The current class instance.
        """
        super().train(mode)
        self.vit.eval()

        return self

    def _get_patch_tokens(self, X: Tensor) -> Tensor:
        return self.vit.patch_embed(X)

    def _prepend_cls_token(self, token_sequence: Tensor) -> Tensor:
        assert self.vit.cls_token is not None

        batch_size = token_sequence.shape[0]
        cls_token_batch = self.vit.cls_token.expand(batch_size, -1, -1)

        return torch.cat((cls_token_batch, token_sequence), dim=1)

    def _add_positional_embeddings(self, token_sequence: Tensor) -> Tensor:
        assert self.vit.pos_embed is not None
        return token_sequence + self.vit.pos_embed

    def _pass_through_transformer(self, token_sequence: Tensor) -> Tensor:
        for block in self.vit.blocks:
            token_sequence = block(token_sequence)

        return token_sequence

    def _normalize_tokens(self, token_sequence: Tensor) -> Tensor:
        return self.vit.norm(token_sequence)


In [ ]:
class L2PModel(nn.Module):
    """
    The complete pipeline of the forward pass.
    """

    def __init__(
        self,
        num_classes: int,
        pool_size: int,
        prompt_length: int,
        top_n: int,
        use_frequency_penalty: bool,
        embed_dimension: int = 768,
        key_dimension: int = 768,
    ) -> None:
        super().__init__()

        self.feature_extractor = ViTFeatureExtractor()

        self.prompt_pool = PromptPool(
            pool_size=pool_size,
            prompt_length=prompt_length,
            embed_dimension=embed_dimension,
            key_dimension=key_dimension,
            top_n=top_n,
            use_frequency_penalty=use_frequency_penalty,
        )

        self.prompt_token_classifier = ClassificationHead(
            embed_dimension=embed_dimension,
            num_classes=num_classes,
            prompt_length=prompt_length,
            top_n=top_n,
        )

    def classify_and_calculate_surrogate_loss_with_L2P(
        self,
        X: Tensor,
        queries: Tensor,
    ) -> tuple[Tensor, Tensor]:
        """
        Image classification pipeline:
        - Uses the precomputed queries to select prompts and calculate the surrogate loss.
        - Embeds image with CLS tokens and positional embeddings.
        - Prepends prompts to the result.
        - Passes the tokens of the previous step through the transformer layer.
        - Extracts the transformed prompt tokens to classify and return logits.

        Args:
            X (Tensor): Input features.
            queries (Tensor): Precomputed CLS tokens.

        Returns:
            tuple[Tensor, Tensor]: Logits and surrogate loss.
        """
        prompts, surrogate_loss = (
            self.prompt_pool.select_prompts_and_calculate_loss_from_queries(queries)
        )

        input_tokens = self._build_input_tokens_with_prompts(X, prompts)

        transformer_output = self.feature_extractor.transform_and_normalize_tokens(
            input_tokens
        )
        logits = (
            self.prompt_token_classifier.extract_transformed_prompt_tokens_and_classify(
                transformer_output
            )
        )

        return logits, surrogate_loss

    def _build_input_tokens_with_prompts(self, X: Tensor, prompts: Tensor) -> Tensor:
        input_tokens_without_prompts = (
            self.feature_extractor.embed_and_build_input_tokens(X)
        )
        input_tokens = self._combine_tokens(prompts, input_tokens_without_prompts)

        return input_tokens

    def _combine_tokens(self, prompts: Tensor, base_tokens: Tensor) -> Tensor:
        return torch.cat((prompts, base_tokens), dim=1)


 ## 2.2. Train

In [ ]:
class ContinualLearningEvaluator:
    """
    Calculates model accuracy on tasks.
    """

    def __init__(
        self,
        model: L2PModel,
        batch_size: int,
        device: device | str,
        shared_test_set: bool,
    ) -> None:
        self.model = model
        self.batch_size = batch_size
        self.device = device
        self.shared_test_set = shared_test_set

    def calculate_accuracies_on_each_task(self, tasks: list[Task]) -> dict[int, float]:
        """
        Calculates and records average accuracies evaluated on each input tasks with the current model.

        Args:
            tasks (list[Task]): Input tasks.

        Returns:
            dict[int, float]: Recorded average accuracies.
        """
        self.model.eval()

        if self.shared_test_set:
            task_accuracy = self._calculate_average_accuracy_on_task(tasks[0])
            task_accuracies = {
                task_index: task_accuracy for task_index in range(len(tasks))
            }

        else:
            task_accuracies = {}
            for current_task_index, current_task in enumerate(tasks):
                current_task_accuracy = self._calculate_average_accuracy_on_task(
                    current_task
                )
                task_accuracies[current_task_index] = current_task_accuracy

                print(f"Accuracy on task {current_task_index}: {current_task_accuracy}")

        return task_accuracies

    def _calculate_average_accuracy_on_task(self, task: Task) -> float:
        """
        Calculates average accuracy on a task by dividing the total correct predictions over the number of samples.

        Args:
            task (Task): Task.

        Returns:
            float: Average accuracy.
        """
        batch_iterator = task.get_test_batches(self.batch_size)

        total_correct_predictions = torch.zeros(
            (), device=self.device, dtype=torch.long
        )
        total_samples = 0

        with torch.inference_mode():
            for batch in batch_iterator:
                correct_predictions, samples = (
                    self._get_correct_predictions_and_total_samples_from_batch(batch)
                )
                total_correct_predictions += correct_predictions
                total_samples += samples

        if total_samples == 0:
            return 0.0

        task_accuracy = total_correct_predictions.item() / total_samples

        return task_accuracy

    def _get_correct_predictions_and_total_samples_from_batch(
        self, batch: Batch
    ) -> tuple[Tensor, int]:
        batch.to(self.device)

        X = batch.get_X()
        Y = batch.get_Y()
        queries = batch.get_queries()

        with torch.autocast("cuda", dtype=torch.float16):
            logits, _ = self.model.classify_and_calculate_surrogate_loss_with_L2P(
                X, queries
            )
        predictions = torch.argmax(logits.float(), dim=1)

        num_correct_predictions = (predictions == Y).sum()
        total_samples = Y.shape[0]

        return num_correct_predictions, total_samples


In [ ]:
class MetricsTracker:
    """
    Records a table accuracies where accuracies[i][j] is the accuracy of the model evaluated on task j after just having trained on task i.
    """

    def __init__(self, num_tasks: int) -> None:
        self.num_tasks = num_tasks
        self.accuracies = [[0.0 for _ in range(num_tasks)] for _ in range(num_tasks)]

    def record_accuracy_on_taskB_after_training_on_taskA(
        self, taskA: int, taskB: int, accuracy: float
    ) -> None:
        self.accuracies[taskA][taskB] = accuracy

    def get_average_accuracy_on_all_tasks_including_current(
        self,
        current_trained_task_index: int,
    ) -> float:
        """
        Calculates the average accuracy evaluated on all tasks up to the current task index (inclusive) after having trained on the current task.

        Args:
            current_trained_task_index (int): Index of the most recently trained on task.

        Returns:
            float: Average accuracy.
        """
        total_accuracy = 0.0
        num_seen_tasks = current_trained_task_index + 1

        for evaluated_task_index in range(num_seen_tasks):
            total_accuracy += self.accuracies[current_trained_task_index][
                evaluated_task_index
            ]

        return total_accuracy / num_seen_tasks

    def get_average_forgetting_on_all_tasks_so_far(
        self,
        current_trained_task_index: int,
    ) -> float:
        """
        Calculates the average of all the forgetting values on all past tasks after having trained on the current task.

        Args:
            current_trained_task_index (int): Index of the most recently trained on task.

        Returns:
            float: Average forgetting value.
        """
        if current_trained_task_index == 0:
            return 0.0

        forgetting_values = self._get_forgetting_values_so_far(
            current_trained_task_index
        )

        return sum(forgetting_values) / len(forgetting_values)

    def _get_forgetting_values_so_far(
        self,
        current_trained_task_index: int,
    ) -> list[float]:
        """
        Gets all the forgetting values on all past tasks after having trained on the current task.

        Args:
            current_trained_task_index (int): Index of the most recently trained on task.

        Returns:
            list[float]: Forgetting values.
        """
        return [
            self._get_forgetting_on_taskA_after_training_on_task_B(
                taskA_index=evaluated_task_index,
                taskB_index=current_trained_task_index,
            )
            for evaluated_task_index in range(current_trained_task_index)
        ]

    def _get_forgetting_on_taskA_after_training_on_task_B(
        self,
        taskA_index: int,
        taskB_index: int,
    ) -> float:
        """
        Calculates forgetting value on task A after having trained on task B, which is the difference between the peak accuracy ever evaluated on task A and the current model's accuracy on task A.

        Args:
            taskA_index (int): Index of task A.
            taskB_index (int): Index of task B.

        Returns:
            float: Forgetting value.
        """
        peak_accuracy = self._get_peak_accuracy_on_task_A_after_training_up_to_task_B(
            taskA_index=taskA_index,
            taskB_index=taskB_index - 1,
        )
        current_accuracy = self.accuracies[taskB_index][taskA_index]

        return peak_accuracy - current_accuracy

    def _get_peak_accuracy_on_task_A_after_training_up_to_task_B(
        self, taskA_index: int, taskB_index: int
    ) -> float:
        """
        Gets the peak accuracy ever evaluated on task A up to the point of having trained on task B.

        Args:
            taskA_index (int): Index of task A.
            taskB_index (int): Index of task B.

        Returns:
            float: Peak accuracy.
        """
        peak_accuracy = 0.0

        for trained_task_index in range(taskA_index, taskB_index + 1):
            accuracy = self.accuracies[trained_task_index][taskA_index]

            if accuracy > peak_accuracy:
                peak_accuracy = accuracy

        return peak_accuracy



In [ ]:
class OptimizerFactory:
    """
    Creates an Adam optimizer for the trainable parameters of the model.
    """

    def __init__(self, learning_rate: float) -> None:
        self.learning_rate = learning_rate

    def create_optimizer(self, model: L2PModel) -> Optimizer:
        trainable_parameters = self._get_trainable_parameters(model)

        return Adam(trainable_parameters, lr=self.learning_rate)

    def _get_trainable_parameters(self, model: L2PModel) -> list[Parameter]:
        return [
            parameter for parameter in model.parameters() if parameter.requires_grad
        ]


In [ ]:
class QueryPrecomputer:
    """
    Runs the frozen query ViT once over the data of every task and stores the resulting queries (CLS tokens) inside the tasks.
    """

    def __init__(
        self,
        feature_extractor: ViTFeatureExtractor,
        batch_size: int,
        device: device | str,
    ) -> None:
        self.feature_extractor = feature_extractor
        self.batch_size = batch_size
        self.device = device

    def precompute_for_tasks(self, tasks: list[Task], shared_test_set: bool) -> None:
        """
        Extracts CLS tokens from each batch in the task and caches them inside the task.

        Args:
            tasks (list[Task]): Tasks.
            shared_test_set (bool): Whether every task uses the same test split. This is only true for CORe50.
        """
        for task in tasks:
            task.set_queries(
                "train", self._extract_cls_tokens_from_batches(task, "train")
            )

        if shared_test_set:
            shared_queries = self._extract_cls_tokens_from_batches(tasks[0], "test")
            for task in tasks:
                task.set_queries("test", shared_queries)
        else:
            for task in tasks:
                task.set_queries(
                    "test", self._extract_cls_tokens_from_batches(task, "test")
                )

    def _extract_cls_tokens_from_batches(self, task: Task, split: str) -> Tensor:
        with torch.no_grad():
            batches = task.get_batches(split, self.batch_size, shuffle=False)

            return torch.cat(
                [
                    self.feature_extractor.pass_through_model_and_extract_cls_tokens(
                        batch.to(self.device).get_X()
                    ).cpu()
                    for batch in batches
                ]
            )



In [ ]:
class RehearsalBuffer:
    def __init__(
        self,
        samples_per_class: int,
        random_seed: int,
        allow_class_overlap: bool,
    ) -> None:
        self.samples_per_class = samples_per_class
        self.buffer: Batch | None = None
        self.recorded_classes: set[int] = set()
        self.random_seed = random_seed
        self.generator = torch.Generator().manual_seed(self.random_seed)
        self.allow_class_overlap = allow_class_overlap

    def get_recorded_classes(self) -> set[int]:
        return self.recorded_classes

    def is_empty(self) -> bool:
        return self.buffer is None

    def fetch_and_save_samples_from_task(self, task: Task) -> None:
        """
        Fetch a fixed number of samples per class from the task and add them to the buffer.

        Args:
            task (Task): Task.
        """
        class_ids_in_task = task.get_class_ids()

        if not self.allow_class_overlap:
            assert self.recorded_classes.isdisjoint(class_ids_in_task)

        batch_to_buffer = task.get_n_samples_per_class(self.samples_per_class)

        self.recorded_classes.update(class_ids_in_task)
        self._add_samples(batch_to_buffer)

    def get_random_samples(self, batch_size: int) -> Batch:
        """
        Gets random samples from the rehearsal buffer, packing them into a batch.

        Args:
            batch_size (int): Output batch size.

        Returns:
            Batch: Packaged random samples.
        """
        assert self.buffer is not None
        buffer_size = self.buffer.get_batch_size()
        random_indices = torch.randperm(buffer_size, generator=self.generator)[
            : min(batch_size, buffer_size)
        ]

        return self.buffer.select(random_indices)

    def _add_samples(self, batch_to_buffer: Batch) -> None:
        if self.buffer is None:
            self.buffer = batch_to_buffer
        else:
            self.buffer = self.buffer.merge(batch_to_buffer)



In [ ]:
class SingleTaskTrainer:
    """
    Trains the model on one task at a time, updating the trainable components: prompt vectors, prompt keys and classification head.
    """

    def __init__(
        self,
        model: L2PModel,
        optimizer_factory: OptimizerFactory,
        epochs_per_task: int,
        batch_size: int,
        device: device | str,
        lambda_surrogate: float,
        grad_clip_max_norm: float,
        rehearsal_buffer: RehearsalBuffer | None,
    ) -> None:
        self.model = model
        self.optimizer_factory = optimizer_factory
        self.epochs_per_task = epochs_per_task
        self.batch_size = batch_size
        self.device = device
        self.lambda_surrogate = lambda_surrogate
        self.grad_clip_max_norm = grad_clip_max_norm
        self.rehearsal_buffer = rehearsal_buffer
        self.scaler = torch.amp.GradScaler("cuda")

    def train_on_task(self, task: Task, total_num_classes: int) -> list[float]:
        """
        Performs training on one task for a predetermined number of epochs and records all the epoch losses. Initialize a new optimizer for each task.

        Args:
            task (Task): Task.
            total_num_classes (int): Total number of classes in the dataset.

        Returns:
            list[float]: Epoch losses.
        """
        self.model.train()

        # Optimizer created anew, internal states are reset every task.
        optimizer = self.optimizer_factory.create_optimizer(self.model)

        current_task_class_mask = self._build_class_mask(
            task.get_class_ids(), total_num_classes
        )
        rehearsal_class_mask = self._build_rehearsal_class_mask(total_num_classes)

        batch_iterator = task.get_train_batches(self.batch_size)

        epoch_losses = self._train_on_all_epochs_and_record_losses(
            batch_iterator,
            current_task_class_mask,
            rehearsal_class_mask,
            optimizer,
        )

        return epoch_losses

    def _train_on_all_epochs_and_record_losses(
        self,
        batch_iterator: BatchIterator,
        current_task_class_mask: Tensor,
        rehearsal_class_mask: Tensor | None,
        optimizer: Optimizer,
    ) -> list[float]:
        """
        Trains the model in multiple epochs and records all the epoch losses.

        Args:
            batch_iterator (BatchIterator): Iterator for all the batches in one task.
            current_task_class_mask (Tensor): Current task's class labels mask.
            rehearsal_class_mask (Tensor | None): Rehearsal buffer class labels mask.
            optimizer (Optimizer): Optimizer (Adam).

        Returns:
            list[float]: Epoch losses.
        """
        epoch_losses = []

        for epoch_index in range(self.epochs_per_task):
            print(f"** EPOCH {epoch_index} **")

            epoch_loss = self._train_on_epoch_and_calculate_average_loss(
                batch_iterator,
                current_task_class_mask,
                rehearsal_class_mask,
                optimizer,
            )
            epoch_losses.append(epoch_loss)

        return epoch_losses

    def _train_on_epoch_and_calculate_average_loss(
        self,
        batch_iterator: BatchIterator,
        current_task_class_mask: Tensor,
        rehearsal_class_mask: Tensor | None,
        optimizer: Optimizer,
    ) -> float:
        """
        Trains the model on every batch in one task in one epoch. Then calculates average loss.

        Args:
            batch_iterator (BatchIterator): Iterator for all the batches in one task.
            current_task_class_mask (Tensor): Current task's class labels mask.
            rehearsal_class_mask (Tensor | None): Rehearsal buffer class labels mask.
            optimizer (Optimizer): Optimizer (Adam).

        Returns:
            float: Average epoch loss.
        """
        total_epoch_loss = torch.zeros((), device=self.device)

        for batch_index, batch in enumerate(batch_iterator):
            batch_loss = self._train_on_batch(
                batch,
                current_task_class_mask,
                rehearsal_class_mask,
                optimizer,
            )
            total_epoch_loss += batch_loss

            if batch_index % 10 == 0 or batch_index == len(batch_iterator) - 1:
                print(f"Batch {batch_index} loss: {batch_loss.item()}")

        num_batches = len(batch_iterator)
        assert num_batches > 0
        average_epoch_loss = (total_epoch_loss / num_batches).item()

        return average_epoch_loss

    def _train_on_batch(
        self,
        batch: Batch,
        current_task_class_mask: Tensor,
        rehearsal_class_mask: Tensor | None,
        optimizer: Optimizer,
    ) -> Tensor:
        """
        Trains data on a batch in the following steps:
        - Moves data to desired device, clear gradients.
        - If rehearsal is used and not empty, samples a batch with the same batch size from it. The current batch and the rehearsal batch are kept as two separate chunks.
        - Forwards the data through the complete L2P model, receiving classification logits and surrogate loss.
        - Masks each chunk's logits with its own class mask.
        - Calculates cross entropy loss from logits and chunk labels.
        - Calculates total loss and scales it by the chunk's portion of total samples, then performs backpropagation, clips gradients and updates parameters.

        Args:
            batch (Batch): Batch.
            current_task_class_mask (Tensor): Current task's class labels mask.
            rehearsal_class_mask (Tensor | None): Rehearsal buffer class labels mask.
            optimizer (Optimizer): Optimizer (Adam).

        Returns:
            Tensor: Total loss value.
        """
        batch.to(self.device)
        self._clear_optimizer_gradients(optimizer)

        chunks_of_batch_and_mask = [(batch, current_task_class_mask)]

        if rehearsal_class_mask is not None:
            assert self.rehearsal_buffer is not None
            rehearsal_batch = self.rehearsal_buffer.get_random_samples(
                self.batch_size
            ).to(self.device)
            chunks_of_batch_and_mask.append((rehearsal_batch, rehearsal_class_mask))

        total_samples = sum(
            chunk_batch.get_batch_size() for chunk_batch, _ in chunks_of_batch_and_mask
        )

        total_loss_value = torch.zeros((), device=self.device)
        for chunk_batch, chunk_class_mask in chunks_of_batch_and_mask:
            X_chunk = chunk_batch.get_X()
            Y_chunk = chunk_batch.get_Y()
            queries_chunk = chunk_batch.get_queries()

            with torch.autocast("cuda", dtype=torch.float16):
                logits, surrogate_loss = (
                    self.model.classify_and_calculate_surrogate_loss_with_L2P(
                        X_chunk, queries_chunk
                    )
                )
            masked_logits = self._mask_logits_with_class_mask(
                logits.float(), chunk_class_mask
            )

            cross_entropy_loss = nn.functional.cross_entropy(masked_logits, Y_chunk)
            chunk_loss = cross_entropy_loss + (self.lambda_surrogate * surrogate_loss)

            chunk_weight = X_chunk.shape[0] / total_samples
            self._backpropagation(loss_result=chunk_loss * chunk_weight)

            total_loss_value += (chunk_loss * chunk_weight).detach()

        self.scaler.unscale_(optimizer)
        self._clip_gradients()

        self._update_parameters_using_optimizer(optimizer)
        self.scaler.update()

        return total_loss_value

    def _clip_gradients(self) -> None:
        trainable_parameters = [
            parameter
            for parameter in self.model.parameters()
            if parameter.requires_grad
        ]
        torch.nn.utils.clip_grad_norm_(
            trainable_parameters, max_norm=self.grad_clip_max_norm
        )

    def _build_rehearsal_class_mask(self, total_num_classes: int) -> Tensor | None:
        if self.rehearsal_buffer and not self.rehearsal_buffer.is_empty():
            rehearsal_class_ids = self.rehearsal_buffer.get_recorded_classes()
            rehearsal_class_mask = self._build_class_mask(
                list(rehearsal_class_ids), total_num_classes
            )
        else:
            rehearsal_class_mask = None

        return rehearsal_class_mask

    def _build_class_mask(self, class_ids: list[int], total_num_classes: int) -> Tensor:
        mask = torch.zeros(total_num_classes, dtype=torch.bool, device=self.device)
        mask[class_ids] = True

        return mask

    def _backpropagation(self, loss_result: Tensor) -> None:
        self.scaler.scale(loss_result).backward()

    def _update_parameters_using_optimizer(self, optimizer: Optimizer) -> None:
        self.scaler.step(optimizer)

    def _clear_optimizer_gradients(self, optimizer: Optimizer) -> None:
        optimizer.zero_grad()

    def _mask_logits_with_class_mask(
        self, logits: Tensor, class_mask: Tensor
    ) -> Tensor:
        inverted_mask = ~class_mask.unsqueeze(0)
        masked_logits = logits.masked_fill(inverted_mask, float("-inf"))

        return masked_logits


In [ ]:
class ContinualLearningPipeline:
    """
    The complete L2P training and evaluating pipeline.
    """

    def __init__(
        self,
        model: L2PModel,
        data_package: DataPackage,
        single_task_trainer: SingleTaskTrainer,
        evaluator: ContinualLearningEvaluator,
        metrics_tracker: MetricsTracker,
        rehearsal_buffer: RehearsalBuffer | None,
    ) -> None:
        self.model = model
        self.data_package = data_package
        self.single_task_trainer = single_task_trainer
        self.evaluator = evaluator
        self.metrics_tracker = metrics_tracker
        self.rehearsal_buffer = rehearsal_buffer

    def train_on_dataset_for_accuracy_and_forgetting(self) -> None:
        """
        Performs training on all tasks of the dataset, at each task:
        - Trains on task and updates parameters.
        - Evaluates current model on all tasks seen including the current one.
        - Records accuracies obtained in the previous step.
        - If prompt frequency penalty is used, updates prompt frequency table.
        - Logs all the accuracies and forgetting values evaluated on all tasks so far using W&B.

        At the end, logs the final model's average accuracy on all tasks, as well as average forgetting value across all tasks.
        """
        num_tasks = self.data_package.get_number_of_tasks()

        for current_task_index in range(num_tasks):
            print(f"----- TASK {current_task_index} -----")

            print("=== TRAINING ===")
            self._train_on_current_task_and_update_rehearsal(current_task_index)

            print("=== EVALUATING ===")
            accuracies_of_all_tasks_up_to_current = (
                self._calculate_accuracies_on_all_tasks_up_to_index(current_task_index)
            )

            self._record_accuracies_after_training_on_current_task(
                accuracies_of_all_tasks_up_to_current,
                current_task_index,
            )

            self._update_prompt_frequencies_if_enabled()

            self._log_accuracies_and_forgetting(
                current_task_index,
                accuracies_of_all_tasks_up_to_current,
                is_final_task=current_task_index == num_tasks - 1,
            )

    def _log_accuracies_and_forgetting(
        self,
        current_task_index: int,
        accuracies_of_all_tasks_up_to_current: dict[int, float],
        is_final_task: bool,
    ) -> None:
        log_data: dict[str, int | float] = {"current_task_index": current_task_index}

        for task_index, accuracy in accuracies_of_all_tasks_up_to_current.items():
            log_data[f"accuracy/task_{task_index}"] = accuracy

            if task_index < current_task_index:
                forgetting = self.metrics_tracker._get_forgetting_on_taskA_after_training_on_task_B(
                    taskA_index=task_index, taskB_index=current_task_index
                )

                print(f"Forgetting on task {task_index}: {forgetting}")
                log_data[f"forgetting/task_{task_index}"] = forgetting

        wandb.log(log_data)

        if is_final_task:
            final_average_accuracy = self.metrics_tracker.get_average_accuracy_on_all_tasks_including_current(
                current_trained_task_index=current_task_index
            )
            final_average_forgetting = (
                self.metrics_tracker.get_average_forgetting_on_all_tasks_so_far(
                    current_trained_task_index=current_task_index
                )
            )

            print("----- FINAL EVALUATION -----")
            print(f"Final average accuracy: {final_average_accuracy}")
            print(f"Final average forgetting: {final_average_forgetting}")

            wandb.summary["final_average_accuracy"] = final_average_accuracy
            wandb.summary["final_average_forgetting"] = final_average_forgetting

    def _train_on_current_task_and_update_rehearsal(
        self, current_task_index: int
    ) -> None:
        current_task = self.data_package.get_task(current_task_index)
        num_classes = self.data_package.get_number_of_classes()

        self.single_task_trainer.train_on_task(current_task, num_classes)

        if self.rehearsal_buffer:
            self.rehearsal_buffer.fetch_and_save_samples_from_task(current_task)

    def _calculate_accuracies_on_all_tasks_up_to_index(
        self,
        current_task_index: int,
    ) -> dict[int, float]:
        print(f"** AFTER TRAINING ON TASK {current_task_index} **")

        task_indices_up_to_current = self._get_all_tasks_up_to_index(current_task_index)
        task_accuracies = self.evaluator.calculate_accuracies_on_each_task(
            task_indices_up_to_current
        )

        return task_accuracies

    def _record_accuracies_after_training_on_current_task(
        self,
        task_accuracies: dict[int, float],
        current_task_index: int,
    ) -> None:
        for evaluated_task_index, accuracy in task_accuracies.items():
            self.metrics_tracker.record_accuracy_on_taskB_after_training_on_taskA(
                taskA=current_task_index,
                taskB=evaluated_task_index,
                accuracy=accuracy,
            )

    def _get_all_tasks_up_to_index(self, current_task_index: int) -> list[Task]:
        seen_tasks = self.data_package.get_tasks()[: current_task_index + 1]

        return seen_tasks

    def _update_prompt_frequencies_if_enabled(self) -> None:
        if self.model.prompt_pool.use_frequency_penalty:
            self.model.prompt_pool.update_frequency_after_task()



 # 3. Experiment

 ## 3.1. Core

In [ ]:
def set_global_seed(random_seed: int) -> None:
    random.seed(random_seed)
    np.random.seed(random_seed)
    torch.manual_seed(random_seed)
    torch.cuda.manual_seed_all(random_seed)



In [ ]:
class ProjectDetail(str, Enum):
    L2P_PROJECT = "L2P_PROJECT"


class GroupDetail(str, Enum):
    OFFICIAL = "OFFICIAL"
    TEST = "TEST"


class EntityDetail(str, Enum):
    DEFAULT = "nhuy165-org"


class DatasetTagDetail(str, Enum):
    CIFAR100 = "CIFAR100"
    GAUSSIAN_CIFAR100 = "GAUSSIAN_CIFAR100"
    CORE50 = "CORE50"
    FIVE_DATASETS = "FIVE_DATASETS"


class ModelTagDetail(str, Enum):
    L2P = "L2P"


class BufferTagDetail(str, Enum):
    BUFFER = "BUFFER"
    NO_BUFFER = "NO_BUFFER"


class AblationTagDetail(str, Enum):
    PROMPT_POOL_ABLATION = "PROMPT_POOL_ABLATION"
    KEY_VALUE_PAIR_ABLATION = "KEY_VALUE_PAIR_ABLATION"
    DIVERSIFIED_SELECTION_ABLATION = "DIVERSIFIED_SELECTION_ABLATION"
    NO_ABLATION = "NO_ABLATION"

class FrequencyTableTagDetail(str, Enum):
    FREQUENCY_TABLE = "FREQUENCY_TABLE"
    NO_FREQUENCY_TABLE = "NO_FREQUENCY_TABLE"


In [ ]:
class ExperimentRunner:
    FETCHER_MAP: ClassVar[dict[FetcherType, type[Fetcher]]] = {
        FetcherType.CIFAR100: CIFARFetcher,
        FetcherType.CORE50: CORe50Fetcher,
        FetcherType.FIVE_DATASETS: FiveDatasetsFetcher,
        FetcherType.GAUSSIAN_CIFAR100: GaussianCIFARFetcher,
    }

    def __init__(
        self,
        random_seed: int,
        device: device | str,
        data_fetcher: FetcherType,
        prompt_pool_size: int,
        prompt_length: int,
        top_n_prompts: int,
        use_frequency_penalty: bool,
        batch_size: int,
        epochs_per_task: int,
        learning_rate: float,
        lambda_surrogate: float,
        grad_clip_max_norm: float,
        rehearsal_samples_per_class: int,
        wandb_log_details: dict[str, Any],
    ) -> None:
        self.random_seed = random_seed
        self.device = device
        self.data_fetcher = data_fetcher
        self.prompt_pool_size = prompt_pool_size
        self.prompt_length = prompt_length
        self.top_n_prompts = top_n_prompts
        self.use_frequency_penalty = use_frequency_penalty
        self.batch_size = batch_size
        self.epochs_per_task = epochs_per_task
        self.learning_rate = learning_rate
        self.lambda_surrogate = lambda_surrogate
        self.grad_clip_max_norm = grad_clip_max_norm
        self.rehearsal_samples_per_class = rehearsal_samples_per_class
        self.shared_test_set = self.data_fetcher == FetcherType.CORE50
        self.wandb_log_details = wandb_log_details

        self.config = {
            "random_seed": self.random_seed,
            "device": self.device,
            "data_fetcher": self.data_fetcher,
            "prompt_pool_size": self.prompt_pool_size,
            "prompt_length": self.prompt_length,
            "top_n_prompts": self.top_n_prompts,
            "use_frequency_penalty": self.use_frequency_penalty,
            "batch_size": self.batch_size,
            "epochs_per_task": self.epochs_per_task,
            "learning_rate": self.learning_rate,
            "lambda_surrogate": self.lambda_surrogate,
            "grad_clip_max_norm": self.grad_clip_max_norm,
            "rehearsal_samples_per_class": self.rehearsal_samples_per_class,
            "shared_test_set": self.shared_test_set,
        }

        self.pipeline = self._initialize()

    def execute(self) -> None:
        """
        Executes the training and logs the result to W&B, as well as printing it to the terminal.
        """
        try:
            self.pipeline.train_on_dataset_for_accuracy_and_forgetting()
        finally:
            wandb.finish()

    def _initialize(self) -> ContinualLearningPipeline:
        """
        Initializes all the necessary components before assembling the pipeline based on the passed configurations.
        W&B is initialized before the initialization.

        Returns:
            ContinualLearningPipeline: Pipeline.
        """

        self.wandb_log_details.update({"config": self.config})
        wandb.init(**self.wandb_log_details)

        wandb.define_metric("current_task_index")
        wandb.define_metric("accuracy/*", step_metric="current_task_index")
        wandb.define_metric("forgetting/*", step_metric="current_task_index")

        set_global_seed(random_seed=self.random_seed)

        data_package = self._initialize_data()

        num_classes = data_package.get_number_of_classes()
        L2P_model = self._intialize_model(num_classes=num_classes)

        self._precompute_queries(model=L2P_model, data_package=data_package)

        optimizer_factory = self._initialize_optimizer_factory()

        num_tasks = data_package.get_number_of_tasks()
        rehearsal_buffer = self._initialize_rehearsal_buffer(num_tasks=num_tasks)

        single_task_trainer = self._intialize_single_task_trainer(
            model=L2P_model,
            optimizer_factory=optimizer_factory,
            rehearsal_buffer=rehearsal_buffer,
        )

        evaluator = self._initialize_evaluator(model=L2P_model)

        metrics_tracker = self._intialize_metrics_tracker(num_tasks=num_tasks)

        pipeline = self._initialize_pipeline(
            model=L2P_model,
            data_package=data_package,
            single_task_trainer=single_task_trainer,
            evaluator=evaluator,
            metrics_tracker=metrics_tracker,
            rehearsal_buffer=rehearsal_buffer,
        )

        return pipeline

    def _precompute_queries(self, model: L2PModel, data_package: DataPackage) -> None:
        query_precomputer = QueryPrecomputer(
            feature_extractor=model.feature_extractor,
            batch_size=self.batch_size,
            device=self.device,
        )
        query_precomputer.precompute_for_tasks(
            tasks=data_package.get_tasks(),
            shared_test_set=self.shared_test_set,
        )

    def _initialize_data(self) -> DataPackage:
        data_fetcher_object = self.FETCHER_MAP[self.data_fetcher](
            random_seed=self.random_seed
        )

        data_package = data_fetcher_object.get_data()

        return data_package

    def _intialize_model(self, num_classes: int) -> L2PModel:
        L2P_model = L2PModel(
            num_classes=num_classes,
            pool_size=self.prompt_pool_size,
            prompt_length=self.prompt_length,
            top_n=self.top_n_prompts,
            use_frequency_penalty=self.use_frequency_penalty,
        ).to(self.device)

        return L2P_model

    def _initialize_optimizer_factory(self) -> OptimizerFactory:
        optimizer_factory = OptimizerFactory(
            learning_rate=self.learning_rate,
        )

        return optimizer_factory

    def _initialize_rehearsal_buffer(self, num_tasks: int) -> RehearsalBuffer | None:
        if self.rehearsal_samples_per_class > 0:
            samples_per_class = self.rehearsal_samples_per_class
            if self.shared_test_set:
                samples_per_class = max(1, samples_per_class // num_tasks)
            rehearsal_buffer = RehearsalBuffer(
                samples_per_class=samples_per_class,
                random_seed=self.random_seed,
                allow_class_overlap=self.shared_test_set,
            )
        else:
            rehearsal_buffer = None

        return rehearsal_buffer

    def _intialize_single_task_trainer(
        self,
        model: L2PModel,
        optimizer_factory: OptimizerFactory,
        rehearsal_buffer: RehearsalBuffer | None,
    ) -> SingleTaskTrainer:
        single_task_trainer = SingleTaskTrainer(
            model=model,
            optimizer_factory=optimizer_factory,
            epochs_per_task=self.epochs_per_task,
            batch_size=self.batch_size,
            device=self.device,
            lambda_surrogate=self.lambda_surrogate,
            grad_clip_max_norm=self.grad_clip_max_norm,
            rehearsal_buffer=rehearsal_buffer,
        )

        return single_task_trainer

    def _initialize_evaluator(self, model: L2PModel) -> ContinualLearningEvaluator:
        evaluator = ContinualLearningEvaluator(
            model=model,
            batch_size=self.batch_size,
            device=self.device,
            shared_test_set=self.shared_test_set,
        )

        return evaluator

    def _intialize_metrics_tracker(self, num_tasks: int) -> MetricsTracker:
        metrics_tracker = MetricsTracker(num_tasks=num_tasks)

        return metrics_tracker

    def _initialize_pipeline(
        self,
        model: L2PModel,
        data_package: DataPackage,
        single_task_trainer: SingleTaskTrainer,
        evaluator: ContinualLearningEvaluator,
        metrics_tracker: MetricsTracker,
        rehearsal_buffer: RehearsalBuffer | None,
    ) -> ContinualLearningPipeline:
        pipeline = ContinualLearningPipeline(
            model=model,
            data_package=data_package,
            single_task_trainer=single_task_trainer,
            evaluator=evaluator,
            metrics_tracker=metrics_tracker,
            rehearsal_buffer=rehearsal_buffer,
        )

        return pipeline


 ## 3.2. Run

 ### 3.2.1. CIFAR100

In [ ]:
def get_CIFAR100_config(run_name: str) -> dict[str, Any]:
    wandb_log_details = {
        "project": ProjectDetail.L2P_PROJECT,
        "name": run_name,
        "group": GroupDetail.TEST,
        "entity": EntityDetail.DEFAULT,
        "tags": [
            DatasetTagDetail.CIFAR100,
            ModelTagDetail.L2P,
            BufferTagDetail.NO_BUFFER,
            AblationTagDetail.NO_ABLATION,
            FrequencyTableTagDetail.NO_FREQUENCY_TABLE,
        ],
        "save_code": True,
    }

    run_config = {
        "random_seed": 42,
        "device": "cuda",
        "data_fetcher": FetcherType.CIFAR100,
        "prompt_pool_size": 10,
        "prompt_length": 5,
        "top_n_prompts": 5,
        "use_frequency_penalty": False,
        "batch_size": 128,
        "epochs_per_task": 5,
        "learning_rate": 0.03,
        "lambda_surrogate": 0.5,
        "grad_clip_max_norm": 1.0,
        "rehearsal_samples_per_class": 0,
        "wandb_log_details": wandb_log_details,
    }

    return run_config



In [ ]:
def get_5_datasets_config(run_name: str) -> dict[str, Any]:
    wandb_log_details = {
        "project": ProjectDetail.L2P_PROJECT,
        "name": run_name,
        "group": GroupDetail.TEST,
        "entity": EntityDetail.DEFAULT,
        "tags": [
            DatasetTagDetail.FIVE_DATASETS,
            ModelTagDetail.L2P,
            BufferTagDetail.NO_BUFFER,
            AblationTagDetail.NO_ABLATION,
            FrequencyTableTagDetail.FREQUENCY_TABLE,
        ],
        "save_code": True,
    }

    run_config = {
        "random_seed": 42,
        "device": "cuda",
        "data_fetcher": FetcherType.FIVE_DATASETS,
        "prompt_pool_size": 20,
        "prompt_length": 5,
        "top_n_prompts": 4,
        "use_frequency_penalty": True,
        "batch_size": 128,
        "epochs_per_task": 5,
        "learning_rate": 0.03,
        "lambda_surrogate": 0.5,
        "grad_clip_max_norm": 1.0,
        "rehearsal_samples_per_class": 0,
        "wandb_log_details": wandb_log_details,
    }

    return run_config




In [ ]:
def get_CORe50_config(run_name: str) -> dict[str, Any]:
    wandb_log_details = {
        "project": ProjectDetail.L2P_PROJECT,
        "name": run_name,
        "group": GroupDetail.TEST,
        "entity": EntityDetail.DEFAULT,
        "tags": [
            DatasetTagDetail.CORE50,
            ModelTagDetail.L2P,
            BufferTagDetail.NO_BUFFER,
            AblationTagDetail.NO_ABLATION,
            FrequencyTableTagDetail.NO_FREQUENCY_TABLE,
        ],
        "save_code": True,
    }

    run_config = {
        "random_seed": 42,
        "device": "cuda",
        "data_fetcher": FetcherType.CORE50,
        "prompt_pool_size": 10,
        "prompt_length": 5,
        "top_n_prompts": 5,
        "use_frequency_penalty": False,
        "batch_size": 128,
        "epochs_per_task": 5,
        "learning_rate": 0.03,
        "lambda_surrogate": 0.5,
        "grad_clip_max_norm": 1.0,
        "rehearsal_samples_per_class": 0,
        "wandb_log_details": wandb_log_details,
    }

    return run_config


In [ ]:
def get_gaussian_CIFAR100_config(run_name: str) -> dict[str, Any]:
    wandb_log_details = {
        "project": ProjectDetail.L2P_PROJECT,
        "name": run_name,
        "group": GroupDetail.TEST,
        "entity": EntityDetail.DEFAULT,
        "tags": [
            DatasetTagDetail.GAUSSIAN_CIFAR100,
            ModelTagDetail.L2P,
            BufferTagDetail.NO_BUFFER,
            AblationTagDetail.NO_ABLATION,
            FrequencyTableTagDetail.NO_FREQUENCY_TABLE,
        ],
        "save_code": True,
    }

    run_config = {
        "random_seed": 42,
        "device": "cuda",
        "data_fetcher": FetcherType.GAUSSIAN_CIFAR100,
        "prompt_pool_size": 10,
        "prompt_length": 5,
        "top_n_prompts": 5,
        "use_frequency_penalty": False,
        "batch_size": 128,
        "epochs_per_task": 1,
        "learning_rate": 0.03,
        "lambda_surrogate": 0.5,
        "grad_clip_max_norm": 1.0,
        "rehearsal_samples_per_class": 0,
        "wandb_log_details": wandb_log_details,
    }

    return run_config



In [ ]:
run_config = get_CIFAR100_config("kaggle_CIFAR100")

experiment_runner = ExperimentRunner(**run_config)

print("========== STARTING RUN ==========")
experiment_runner.execute()
print("========== RUN SUCCESSFUL ==========")